# Chapter 3 examples: infinite horizon and discounting

This notebook runs the examples of
[Chapter 3](https://thduynguyen.github.io/gtsam/chapter03): the endless
track, a robot that keeps moving until its episode ends at random.

GTSAM Copyright 2010, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/thduynguyen/gtsam/blob/feature/semiringfactor/gtsam/semiring/doc/chapter03_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    # The semiring module is not in a released GTSAM yet, so install a wheel
    # built from the feature branch.
    %pip install --quiet numpy pyparsing plotly
    %pip install --quiet --no-index --no-deps --find-links https://github.com/thduynguyen/gtsam/releases/expanded_assets/semiring-wheels gtsam-develop
except ImportError:
    pass  # Not running on Colab, do nothing

In [2]:
import numpy as np
from gtsam import DecisionTreeFactor
from gtsam import SemiringDiscreteFactor, SemiringFactorGraph
from gtsam.symbol_shorthand import A, S

np.set_printoptions(precision=4, suppress=True)

## The endless track (Section 1)

The track and its slippery moves are those of Chapter 1. The robot now keeps
moving. Moving Right costs 1, and every step spent in cell 2, at the charger,
pays 2. After each move the episode continues with probability
$\gamma = 0.9$.

In [3]:
L, R = 0, 1
gamma = 0.9
prior = np.array([0.5, 0.5, 0.0])  # p(s0)
policy = np.full((3, 2), 0.5)  # pi(a | s): a coin flip
dynamics = np.zeros((3, 2, 3))  # p(s' | s, a)
dynamics[0, L], dynamics[0, R] = [1, 0, 0], [0.2, 0.8, 0]
dynamics[1, L], dynamics[1, R] = [0.8, 0.2, 0], [0, 0.2, 0.8]
dynamics[2, L], dynamics[2, R] = [0, 0.8, 0.2], [0, 0, 1]
reward = np.array([[0.0, -1.0], [0.0, -1.0], [2.0, 1.0]])  # r(s, a)

## The unrolled chain (Section 3)

The state-to-state table $P_\pi$ and the expected reward $r_\pi$ of the
policy, then one Bellman backup per extra move.

In [4]:
P_pi = np.einsum("sa,sat->st", policy, dynamics)  # P_pi(s, s')
r_pi = (policy * reward).sum(axis=1)  # r_pi(s)
print("P_pi =\n", P_pi)
print("r_pi =", r_pi)

unrolled = {}
V = np.zeros(3)  # no moves left: nothing to collect
for k in range(1, 101):
    V = r_pi + gamma * P_pi @ V  # one more move
    unrolled[k] = V
for k in [1, 2, 5, 10, 20, 50, 100]:
    print(f"{k:3d} moves: V = {unrolled[k]},  J = {prior @ unrolled[k]:.4f}")

P_pi =
 [[0.6 0.4 0. ]
 [0.4 0.2 0.4]
 [0.  0.4 0.6]]
r_pi = [-0.5 -0.5  1.5]
  1 moves: V = [-0.5 -0.5  1.5],  J = -0.5000
  2 moves: V = [-0.95 -0.23  2.13],  J = -0.5900
  5 moves: V = [-1.109   0.1174  3.0392],  J = -0.4958
 10 moves: V = [-0.8013  0.5206  3.5374],  J = -0.1404
 20 moves: V = [-0.4274  0.8991  3.9204],  J = 0.2358
 50 moves: V = [-0.2334  1.0931  4.1145],  J = 0.4299
100 moves: V = [-0.2248  1.1017  4.123 ],  J = 0.4384


## The fixed point (Section 4)

With infinitely many moves the value satisfies $V = r_\pi + \gamma P_\pi V$,
a linear system.

In [5]:
V = np.linalg.solve(np.eye(3) - gamma * P_pi, r_pi)
Q = reward + gamma * dynamics @ V
J = prior @ V
print("V =", V)
print("Q =\n", Q)
print("J =", J)
assert np.allclose(V, [-0.2248, 1.1017, 4.1231], atol=1e-4)
assert np.isclose(J, 0.4385, atol=1e-4)
assert np.allclose((policy * Q).sum(axis=1), V)  # V is the average of Q

# The unrolled chain approaches it, within the bound gamma^k r_max / (1-gamma).
for k in [10, 20, 50, 100]:
    error = np.abs(unrolled[k] - V).max()
    bound = gamma ** k * np.abs(reward).max() / (1 - gamma)
    print(f"{k:3d} moves: error {error:.5f}, bound {bound:.5f}")
    assert error <= bound

V = [-0.2248  1.1017  4.1231]
Q =
 [[-0.2023 -0.2472]
 [ 0.0365  2.1669]
 [ 3.5354  4.7108]]
J = 0.43846720707442965
 10 moves: error 0.58571, bound 6.97357
 20 moves: error 0.20264, bound 2.43153
 50 moves: error 0.00859, bound 0.10308
100 moves: error 0.00004, bound 0.00053


## Forward messages: the discounted visitation (Section 5)

$d(s)$ is the expected number of steps the robot spends in cell $s$ before
the episode ends. It gives the same $J$ from the other side.

In [6]:
d = np.linalg.solve((np.eye(3) - gamma * P_pi).T, prior)
print("d =", d, " sum =", d.sum())
print("d . r_pi =", d @ r_pi)
assert np.isclose(d.sum(), 1 / (1 - gamma))
assert np.isclose(d @ r_pi, J)

d = [3.8062 3.4746 2.7192]  sum = 10.000000000000014
d . r_pi = 0.4384672070744309


## Discount as termination, on a factor graph (Section 2)

Add a fourth state, "ended". After each move the robot goes there with
probability $1 - \gamma$, and stays there, collecting nothing. No discount
appears anywhere else: this is an ordinary semiring factor graph, and its
expectation is the discounted return.

In [7]:
ENDED = 3
ended_dynamics = np.zeros((4, 2, 4))
ended_dynamics[:3, :, :3] = gamma * dynamics  # continue
ended_dynamics[:3, :, ENDED] = 1 - gamma  # end
ended_dynamics[ENDED, :, ENDED] = 1  # stay ended
ended_reward = np.vstack([reward, [0.0, 0.0]])
ended_policy = np.full((4, 2), 0.5)
ended_prior = np.append(prior, 0.0)

state = lambda t: (S(t), 4)  # (key, cardinality): cell 0, 1, 2 or ended
action = lambda t: (A(t), 2)  # 0 = Left, 1 = Right


def probability(keys, table):
    """Lift a probability table to (p, 0)."""
    return SemiringDiscreteFactor(
        DecisionTreeFactor(keys, np.ravel(table).tolist()))


def value(keys, table):
    """Lift a reward table to (1, r)."""
    return SemiringDiscreteFactor.Reward(
        DecisionTreeFactor(keys, np.ravel(table).tolist()))


def chain(moves):
    """The factor graph of the endless track, cut after some moves."""
    graph = SemiringFactorGraph()
    graph.push_back(probability([state(0)], ended_prior))
    for t in range(moves):
        graph.push_back(probability([state(t), action(t)], ended_policy))
        graph.push_back(probability(
            [state(t), action(t), state(t + 1)], ended_dynamics))
        graph.push_back(value([state(t), action(t)], ended_reward))
    return graph


for moves in [1, 2, 5, 10, 20]:
    expectation = chain(moves).expectation()
    print(f"{moves:3d} moves: graph.expectation() = {expectation:.4f}")
    assert np.isclose(expectation, prior @ unrolled[moves])

  1 moves: graph.expectation() = -0.5000
  2 moves: graph.expectation() = -0.5900
  5 moves: graph.expectation() = -0.4958
 10 moves: graph.expectation() = -0.1404
 20 moves: graph.expectation() = 0.2358
